In [1]:
import os
import pandas as pd


In [2]:
# Source - https://stackoverflow.com/a/50051542
# Posted by Aaron Brock, modified by community. See post 'Timeline' for change history
# Retrieved 2026-02-26, License - CC BY-SA 3.0

def to_csv(df, path):
    df = df.copy()
    # Prepend dtypes to the top of df (from https://stackoverflow.com/a/43408736/7607701)
    df.loc[-1] = df.dtypes
    df.index = df.index + 1
    df.sort_index(inplace=True)
    # Then save it to a csv
    df.to_csv(path, index=False)

def read_csv(path):
    # Read types first line of csv
    dtypes = {key:value for (key,value) in pd.read_csv(path,    
              nrows=1).iloc[0].to_dict().items() if 'date' not in value}

    parse_dates = [key for (key,value) in pd.read_csv(path, 
                   nrows=1).iloc[0].to_dict().items() if 'date' in value]
    # Read the rest of the lines with the types from above
    return pd.read_csv(path, dtype=dtypes, parse_dates=parse_dates, skiprows=[1])

In [3]:
def split_df_by_date(df, cutoff_date , date_col = "date"):
                      
    # Read the csv file
    df = df.copy()

    # The date column should already be a pd timestamp but just in case...
    df[date_col] = pd.to_datetime(df[date_col])

    # Sort by date
    df = df.sort_values(date_col).reset_index(drop=True)

    # Make sure cut off date is also a pd Timestamp
    cutoff_date = pd.Timestamp(cutoff_date)

    # Split
    train_df = df[df[date_col] < cutoff_date].copy()
    test_df = df[df[date_col] >= cutoff_date].copy()

    return train_df, test_df

In [4]:
def split_csv_by_date(input_data_path, cutoff_date, date_col="date"):
    df = read_csv(input_data_path)
    return split_df_by_date(df, cutoff_date, date_col=date_col)

In [5]:
def save_train_test_csv(train_df, test_df, train_path, test_path):
    to_csv(train_df, train_path)
    to_csv(test_df, test_path)

First we will set a fixed cut off date which we will use for each of the processed datasets. 

In [6]:
cut_off = "2025-09-01"

Then we will split 
'data/schedule_data/processed_data/by_day/schedule_data.csv' 
and store the split data in 
'data/schedule_data/train_test_split_data/by_day/schedule_data_train.csv' and 'data/schedule_data/train_test_split_data/by_day/schedule_data_test.csv' respectively.

In [7]:
train_df, test_df = split_csv_by_date(
    input_data_path='../data/schedule_data/processed_data/by_day/schedule_data.csv',
    cutoff_date= cut_off,
    date_col="date"
)

save_train_test_csv(
    train_df,
    test_df,
    train_path="../data/schedule_data/train_test_split_data/by_day/schedule_data_train.csv",
    test_path="../data/schedule_data/train_test_split_data/by_day/schedule_data_test.csv"
)

Now we do the same and split each FILE.csv in 'data/schedule_data/processed_data/by_wekday/' and store the split data as 'data/schedule_data/train_test_split_data/by_weekday/FILE_train.csv' and 'data/schedule_data/train_test_split_data/by_weekday/FILE_test.csv' respectively.

In [8]:

input_dir = "../data/schedule_data/processed_data/by_weekday/"

output_dir = "../data/schedule_data/train_test_split_data/by_weekday"

by_weekday_files = [file for file in os.listdir(input_dir) if file.endswith('.csv')]

for filename in by_weekday_files:
    # List the complete file path for each folder
    input_path = os.path.join(input_dir, filename)

    # Drop the .csv part from the file names 
    base_name = os.path.splitext(filename)[0]

    # Then use base_name to name the train/test files and specify the output paths
    train_path = os.path.join(output_dir, f"{base_name}_train.csv")
    test_path = os.path.join(output_dir, f"{base_name}_test.csv")

    # Split the training and testing sets 
    train_df, test_df = split_csv_by_date(
        input_data_path=input_path,
        cutoff_date=cut_off,
        date_col="date"
    )

    # Save the training and testing sets 
    save_train_test_csv(
        train_df=train_df,
        test_df=test_df,
        train_path=train_path,
        test_path=test_path
    )